# 02 — توكنايزر للمبتدئين  ·  Tokenizers from scratch, for a Masri-reading audience

> *لو إنت بتستخدم نموذج زي ChatGPT أو Claude بالعربي، يمكن لاحظت إن الرد بطيء، أو إن
> الموديل بيقطع الكلمات بشكل غريب. النوتبوك ده بيشرحلك ليه ده بيحصل، وإيه اللي ممكن
> يصلحه.*
>
> If you've used ChatGPT or Claude in Arabic — especially in Masri — you've probably
> noticed that responses are slower than in English, or that the model chops words
> oddly. This notebook walks you through *why* that happens, the algorithm under
> the hood (BPE — Byte Pair Encoding), and what would actually fix it. We build a
> tiny BPE tokenizer from scratch on Masri text and watch it learn.

**Prereqs.** Some Python. No prior ML required.
**Companion.** [`01-tokenizer-comparison-msa-masri.ipynb`](01-tokenizer-comparison-msa-masri.ipynb) —
showed *that* English-only tokenizers charge a 2-3× tax on Arabic. This notebook
shows *why* and *how to build one that doesn't*.

## 1. الإعداد  ·  Setup

In [ ]:
from __future__ import annotations

import json
import os
from collections import Counter
from pathlib import Path

from IPython.display import HTML, display
from transformers import AutoTokenizer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PAIRS_PATH = REPO / "eval" / "prompts" / "msa-masri-pairs-v1.json"
print("repo:", REPO)
print("triples:", PAIRS_PATH.exists(), "←", PAIRS_PATH)

## 2. التوكنايزر بيعمل إيه؟  ·  What does a tokenizer actually do?

النموذج اللغوي ما بيشوفش حروف ولا كلمات — بيشوف **أرقام**. التوكنايزر هو الجزء اللي بيحوّل
النص اللي إنت كاتبه لأرقام صحيحة (IDs)، والعكس.

A language model doesn't see characters or words — it sees **integers**. The
tokenizer is the bit that turns your text into integer IDs (and back). The
pipeline looks like this:

```
"عايز شوية ميه"  ──tokenizer──▶  [4581, 309, 12, 9921]  ──model──▶  logits  ──tokenizer──▶  "ايوه حالًا"
                  ▲                                                                  ▲
              text → ids                                                         ids → text
```

Two things matter for the rest of this notebook:

1. **The vocabulary is fixed at training time.** A tokenizer "knows" a finite set
   of pieces (typically 30k-200k). Anything outside that set is built from smaller
   pieces it does know.
2. **More tokens = more compute.** Cost, latency, and context-window usage all scale
   in token count. So a tokenizer that needs 10 tokens for what another does in 4
   makes the user 2.5× slower / more expensive on every single request.

Let's ground this on a Masri sentence.

In [ ]:
SENTENCE_MASRI = "عايز شوية ميه"   # "I want some water"
SENTENCE_EN    = "I want some water"

gpt2 = AutoTokenizer.from_pretrained("gpt2")

for label, text in [("Masri", SENTENCE_MASRI), ("English", SENTENCE_EN)]:
    ids = gpt2.encode(text)
    pieces = [gpt2.decode([i]) for i in ids]
    print(f"{label:>8s}  text   : {text!r}")
    print(f"{'':>8s}  ids    : {ids}  ({len(ids)} tokens)")
    print(f"{'':>8s}  pieces : {pieces}")
    print()

*ملاحظة:* gpt2 بياخد جملة عربية قصيرة (٣ كلمات) وبيقطعها لـ ١٢-١٣ توكن، بينما الجملة
الإنجليزية المماثلة بتاخد ٤ توكنز بس. ليه؟ لأن gpt2 ما عندوش subwords عربية أصلًا — فبيرجع
لمستوى البايتات (byte-level fallback). الهدف من باقي النوتبوك: نشوف **إزاي** الـ BPE بيتدرب،
ونبني واحد بنفسنا للماصري.

## 3. حلين ساذجين بييجوا في بال أي حد  ·  Two naïve approaches that don't work

قبل ما نوصل لـ BPE، خلينا نشوف ليه الحلول الواضحة بتفشل.

### 3a. مستوى الحرف / البايت  ·  Character / byte level

كل حرف = توكن. الفوكاب صغير جدًا (مثلًا 256 بايت)، بس الجمل بتطول جدًا. ده بالظبط اللي gpt2
بيعمله مع العربي كـ fallback.

In [ ]:
text = "عايز شوية ميه"
chars = list(text)
bytes_utf8 = list(text.encode("utf-8"))
print(f"text         : {text!r}    ({len(text)} chars)")
print(f"as chars     : {chars}")
print(f"as utf-8 bytes ({len(bytes_utf8)}): {bytes_utf8[:20]}...")
print()
print(f"vocab size if char-level : ~few hundred (Arabic letters + punctuation)")
print(f"sequence length cost     : ~1 token per character → 13 tokens here")

**المشكلة:** الموديل بيتعب جدًا في تعلّم العلاقة بين حرف وحرف عشان يفهم كلمة. زي ما تطلب من
حد يقرألك كتاب حرف-حرف. بطيء وغير عملي.

### 3b. مستوى الكلمة  ·  Word level

كل كلمة = توكن. الجمل بتقصر، بس الفوكاب بينفجر — كل صيغة (مفرد، جمع، تنوين، ضمائر متصلة)
بتبقى توكن مستقل، وأي كلمة جديدة (out-of-vocabulary, OOV) بتبقى `[UNK]`.

In [ ]:
import re

# A tiny "word-level vocabulary" built from a tiny corpus
tiny_corpus = [
    "عايز شوية ميه",
    "عايزة شوية ميه",        # feminine — different word
    "هيعوزوا شوية ميه",      # they will want — different word
    "بيشرب الميه",            # drinks the water
    "ما شربش الميه",
]
words = [w for line in tiny_corpus for w in re.split(r"\s+", line) if w]
vocab = sorted(set(words))
print(f"corpus has {len(tiny_corpus)} sentences and {len(words)} word-tokens")
print(f"unique words → vocab : {vocab}")
print()
print("note how ('عايز', 'عايزة', 'هيعوزوا') are 3 separate vocab entries")
print("for the same root √ع-و-ز ('to want'). For real text this explodes fast.")

**المشكلة:** أي صيغة جديدة لكلمة موجودة بتبقى OOV. والعربية صرفيًا غنية جدًا (الجذر + أوزان +
ضمائر متصلة)، فالفوكاب بيكبر بالملايين. مش عملي.

**الحل الوسط:** ابدأ من الحروف، وادمج التتابعات الأكتر شيوعًا في **subwords**. ده هو BPE.

## 4. خوارزمية BPE في جملتين  ·  BPE: the whole algorithm in two sentences

> **في كل خطوة: عُدّ كل زوج حروف متتابعين في كل الكلمات في الكوربس. ادمج الزوج الأكتر
> تكرارًا في توكن واحد. كرّر ٣٠٬٠٠٠-١٠٠٬٠٠٠ مرة (= حجم الفوكاب).**

That's it. The algorithm is older than transformers (it comes from a 1994 data-
compression paper by Philip Gage). The merges it learns become the tokenizer's
vocabulary. High-frequency sequences — common roots, common prefixes/suffixes —
end up as single tokens; rare sequences stay as multi-piece sequences.

Let's *do* it on Masri words, by hand, in pure Python, no library.

## 5. BPE بإيدنا على ٦ كلمات ماصري  ·  BPE by hand on 6 Masri words

كوربسنا التعليمي: أربع صيغ من جذر ع-و-ز ('want') مع تكرارات واقعية:

In [ ]:
# Tiny pedagogical corpus: counts mimic real-text frequency.
# عايز/عايزة/عايزين are super common; هيعوزوا is rarer.
corpus_counts = Counter({
    "عايز":     8,
    "عايزة":    5,
    "عايزين":   4,
    "هيعوزوا":  1,
    "ميه":      6,
    "ميته":     2,
})

# Each word starts as a list of single characters with an end-of-word marker.
END = "</w>"
def word_to_pieces(w: str) -> list[str]:
    return list(w) + [END]

splits = {w: word_to_pieces(w) for w in corpus_counts}
for w, pieces in splits.items():
    print(f"{w:>10s}  ×{corpus_counts[w]}   →  {pieces}")

دلوقتي خد كل الأزواج المتتابعة في كل الكلمات، عُدّهم (مع وزن تكرار الكلمة)، اعرف الزوج الأكتر،
وادمجه. كرّر.

In [ ]:
def get_pair_counts(splits: dict[str, list[str]], counts: Counter) -> Counter:
    pair_counts: Counter = Counter()
    for word, pieces in splits.items():
        freq = counts[word]
        for a, b in zip(pieces, pieces[1:]):
            pair_counts[(a, b)] += freq
    return pair_counts

def merge_pair(splits: dict[str, list[str]], pair: tuple[str, str]) -> dict[str, list[str]]:
    a, b = pair
    new_splits = {}
    for word, pieces in splits.items():
        new_pieces, i = [], 0
        while i < len(pieces):
            if i < len(pieces) - 1 and pieces[i] == a and pieces[i+1] == b:
                new_pieces.append(a + b)
                i += 2
            else:
                new_pieces.append(pieces[i])
                i += 1
        new_splits[word] = new_pieces
    return new_splits

merges: list[tuple[str, str]] = []
for step in range(8):
    pair_counts = get_pair_counts(splits, corpus_counts)
    if not pair_counts:
        break
    best_pair, best_count = pair_counts.most_common(1)[0]
    merges.append(best_pair)
    splits = merge_pair(splits, best_pair)
    joined = "".join(best_pair)
    print(f"step {step+1}: merge {best_pair[0]!r} + {best_pair[1]!r} → {joined!r}  (count={best_count})")
    for w, pieces in splits.items():
        print(f"    {w:>10s}  →  {pieces}")
    print()

*شوف اللي حصل:* بعد كم خطوة، الكلمات اللي كانت ٤-٧ حروف بقت ١-٢ توكن. الجذر ع-ا-ي-ز ظهر
كقطعة واحدة لأنه شائع، والصيغ المختلفة (عايز/عايزة/عايزين) بدأت تشترك في نفس البداية.
ده هو الـ BPE — مش أكتر، مش أقل.

## 6. BPE بمكتبة `tokenizers`  ·  BPE with the real library, on real Masri text

نفس الخوارزمية، بس مكتوبة بـ Rust وبتشتغل على آلاف الجمل في ثواني. هندرّب توكنايزر بسيط
على الـ ٣٠ جملة ماصري من الـ pair set اللي عندنا.

> ملاحظة: ٣٠ جملة كوربس صغير جدًا — عمليًا الـ BPE بيتدرب على جيجابايتات. بس للتعليم،
> ده كفاية عشان نشوف الـ merges بتنتج subwords معقولة.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

triples = json.loads(PAIRS_PATH.read_text())["pairs"]
masri_corpus = [p["masri"] for p in triples]
print(f"training corpus: {len(masri_corpus)} Masri sentences, "
      f"{sum(len(s) for s in masri_corpus)} chars total")
print("first 3:", masri_corpus[:3])

In [ ]:
masri_tok = Tokenizer(BPE(unk_token="[UNK]"))
masri_tok.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=200,
    special_tokens=["[UNK]"],
    min_frequency=1,        # tiny corpus → accept every pair we see
    show_progress=False,
)
masri_tok.train_from_iterator(masri_corpus, trainer=trainer)
print(f"trained! vocab size = {masri_tok.get_vocab_size()}")

نشوف أول ٣٠ merge اتعلمها التوكنايزر. الترتيب = من الأكتر شيوعًا للأقل. لاحظ ظهور قطع
ماصري أصيلة زي `ال` (التعريف)، `بـ` (سابقة المضارع)، `إيه`، `ده`، إلخ.

In [ ]:
import tempfile

# The Rust tokenizer doesn't expose merges as a Python list directly,
# so we save → reload as JSON to inspect them.
with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as f:
    masri_tok.save(f.name)
    cfg = json.loads(Path(f.name).read_text())

merges_learned = cfg["model"]["merges"]
print(f"total merges: {len(merges_learned)}")
print("first 30:")
for i, m in enumerate(merges_learned[:30]):
    a, b = (m[0], m[1]) if isinstance(m, list) else m.split(" ", 1)
    print(f"  {i+1:>3d}: {a!r} + {b!r}  →  {a+b!r}")

## 7. التوكنايزر بتاعك ضد gpt2  ·  Your custom Masri tokenizer vs gpt2

نشوف الفرق العملي. هناخد جملة ماصري مش في الكوربس التدريبي ونعدّ التوكنز في الإتنين.

In [ ]:
HELD_OUT = "أنا عايز كباية شاي بالنعناع دلوقتي"  # "I want a cup of mint tea right now"

ours_ids   = masri_tok.encode(HELD_OUT).ids
ours_pieces = masri_tok.encode(HELD_OUT).tokens
gpt2_ids   = gpt2.encode(HELD_OUT)
gpt2_pieces = [gpt2.decode([i]) for i in gpt2_ids]

print(f"sentence: {HELD_OUT!r}  ({len(HELD_OUT)} chars)")
print()
print(f"  custom (Masri-trained, vocab={masri_tok.get_vocab_size()}): "
      f"{len(ours_ids)} tokens")
print(f"    pieces: {ours_pieces}")
print()
print(f"  gpt2 (English-trained, vocab={gpt2.vocab_size}): "
      f"{len(gpt2_ids)} tokens")
print(f"    pieces (first 20): {gpt2_pieces[:20]}")
print()
ratio = len(gpt2_ids) / len(ours_ids)
print(f"  → gpt2 spends {ratio:.1f}× as many tokens as our 200-token Masri BPE")
print(f"    (and gpt2 has 250× the vocab! more vocab ≠ better, if it's the wrong vocab)")

**الدرس:** توكنايزر صغير (٢٠٠ توكن) مدرّب على ماصري بيهزم gpt2 (٥٠٬٠٠٠ توكن) على ماصري،
لأن الـ ٥٠ ألف بتاعت gpt2 كلها إنجليزي. مش حجم الفوكاب اللي بيهم — اللي بيهم هو **ايه** اللي
في الفوكاب. ده هو السبب الفعلي للـ "tokenizer tax" اللي شفناه في الـ notebook 01.

> *Caveat:* توكنايزرنا اتدرّب على ٣٠ جملة بس، فهيفشل بسرعة على نصوص خارج المجال.
> الفكرة هنا تعليمية. للاستخدام الحقيقي بنحتاج كوربس ميجابايتات (مثلًا
> [`arzwiki`](https://huggingface.co/datasets/wikipedia)).

## 8. خوارزميات تانية بسرعة  ·  The other algorithms, briefly

BPE هي الأشهر، بس مش الوحيدة. الأهم تعرف الفرق بين تلاتة:

| Algorithm | Used by | الفكرة الأساسية | إيه الفرق عمليًا |
|---|---|---|---|
| **BPE** (Byte Pair Encoding) | GPT-2/3/4, LLaMA, Mistral, Pythia | ادمج الزوج الأكتر تكرارًا، كرّر | بيحب prefixes/suffixes شائعة. الأبسط والأسرع تدريبًا. |
| **WordPiece** | BERT, DistilBERT | زي BPE بس بيدمج الزوج اللي بيرفع الـ likelihood الأكتر | بيميل لـ subwords أنضف لغويًا. أبطأ شوية. |
| **Unigram (SentencePiece)** | T5, mT5, ALBERT, mBART | ابدأ بفوكاب كبير، شيل الأقل أهمية لحد ما توصل للحجم المطلوب | بيدي عدة طرق لتقطيع نفس الكلمة (sampling). أحسن للغات المتعددة. |

كلهم في الآخر بيحلوا نفس المشكلة (متوسط بين حروف وكلمات)، بس بأذواق مختلفة. للعربية،
فيه دراسات بتقول إن **Unigram** أحسن قليلًا لأنه بيتعامل مع الصرف الغني بشكل أنعم. للماصري
تحديدًا، الفرق الأهم مش الخوارزمية — الفرق الأهم هو **الكوربس التدريبي**: لازم يحتوي
ماصري بكميات كافية، مش فصحى بس.

## 9. ليه ما ينفعش نـ"نضيف" ماصري لـ gpt2؟  ·  Why you can't just patch an existing tokenizer

ممكن تسأل: طب ما نزوّد كم توكن ماصري على فوكاب gpt2؟ الإجابة القصيرة: ينفع تقنيًا، بس مش
هيشتغل، لأن:

1. **embeddings الموديل بتاعها متعلّمة.** كل توكن في فوكاب gpt2 ليه vector جوّه الموديل
   اتدرّب عليه تريليونات المرات. التوكنز الجديدة هتيجي بـ embeddings عشوائية — الموديل
   ما يعرفش يعمل بيها حاجة.
2. **التوكنز الجديدة هتفضل OOV عمليًا.** حتى لو ضفنا `عايز` كتوكن، الموديل ما اتدرّبش على
   استخدامه في سياق، فهيبقى نظريًا في الفوكاب وغير مفهوم عمليًا.
3. **الحل الفعلي: re-train (أو fine-tune) tokenizer + model مع بعض على كوربس فيه ماصري.**
   ده اللي بيعمله مشاريع زي AraBERT و AraGPT2 و mGPT — فوكاب اتبنى من الأول على نص عربي.

ده اللي بيخلي الـ "tokenizer tax" اللي شفناه في nb01 ضريبة بنيوية: مش ممكن تنفك من غير
ما تتدخل في تدريب الموديل نفسه.

## 10. اللي شوفناه + الخطوة الجاية  ·  Recap and what's next

**شوفنا في النوتبوك ده:**

1. التوكنايزر = نص ↔ أرقام. الفوكاب ثابت بعد التدريب.
2. مستوى الحرف بطيء؛ مستوى الكلمة بينفجر. BPE وسط معقول.
3. خوارزمية BPE في جملة: عُدّ الأزواج، ادمج الأكتر، كرّر.
4. بنينا BPE بإيدنا على ٦ كلمات؛ والأدوات الحقيقية بتعمل نفس الشيء بسرعة.
5. توكنايزر صغير مدرّب على ماصري (٢٠٠ توكن) أحسن من gpt2 (٥٠٬٠٠٠ توكن) على نصوص ماصري —
   لأن **محتوى** الفوكاب أهم من حجمه.
6. WordPiece و Unigram بدائل جيدة، بس المهم هو الكوربس مش الخوارزمية.
7. ما ينفعش نضيف توكنز جديدة لموديل موجود من غير إعادة تدريب.

**الخطوة الجاية في فانوس:** notebook 03 (قادم) — هنشوف **فين** بالظبط في الـ residual stream
بتاع موديل صغير بيظهر إشارة الفصحى-ضد-ماصري، ونتأكد لو ده feature متعلّم فعلًا ولّا مجرد
انعكاس للـ tokenizer tax اللي شفناه هنا.